In [1]:
import pandas as pd
import os
import csv

# Directories
dir_t20s = 't20s_csv/'
dir_it20s = 'it20s_csv/'

# Column names
csv_inning_column_names = [
    'Inning_Number', 'Over.Ball', 'Batting_Team', 'Batsman', 
    'Non_Striker', 'Bowler', 'Runs_off_Bat', 'Extras', 'Wides', 
    'No_Balls', 'Byes', 'Leg_Byes', 'Penalty', 'Kind_of_wicket_If_Any', 
    'Dissmissed_Player_When_Wicket'
]
meta_column = ['city', 'gender', 'date', 'toss_winner', 'toss_decision', 'venue']

# Initialize final dataframe list
final_df_list = []

# Function to process files from a directory
def process_directory(directory):
    for file in os.listdir(directory):
        if file.endswith('.csv'):
            with open(os.path.join(directory, file), 'r') as o:
                line_list = csv.reader(o)
                metas = []
                csvs = []
                teams = []

                for line in line_list:
                    # Look for metadata lines
                    if 'info' in line and 'team' not in line and 'player' not in line and 'players' not in line and 'people' not in line:
                        metas.append(line[1:])
                    # Look specifically for team lines
                    elif 'info' in line and 'team' in line:
                        if len(line) > 2:
                            team_name = line[2]  # The team name should be here
                            teams.append(team_name)
                        else:
                            print(f"Unexpected team line format in file {file}: {line}")
                    # Capture ball-by-ball data
                    elif 'ball' in line:
                        csvs.append(line[1:])

                # Ensure two teams were captured
                if len(teams) != 2:
                    print(f"Team extraction issue in file {file}: {teams}")
                else:
                    metas.append(('Team1', teams[0]))
                    metas.append(('Team2', teams[1]))

                try:
                    # Convert the metadata into a dictionary and filter using meta_column
                    metas_dict = pd.Series(dict(metas)).get(meta_column + ['Team1', 'Team2'], {})

                    # Handle any missing metadata gracefully
                    for col in meta_column + ['Team1', 'Team2']:
                        metas_dict[col] = metas_dict.get(col, None)

                except Exception as E:
                    print("Metadata extraction error for file:", file)
                    print(metas)
                    raise E  # Raise the error to be more descriptive

                # Convert the ball data to a DataFrame
                csv_df = pd.DataFrame(csvs, columns=csv_inning_column_names)
                
                # Add metadata columns to the DataFrame
                for k, v in metas_dict.items():
                    csv_df[k] = v
                
                # Add match_id column
                csv_df['match_id'] = file  # Just the file name, not the full path
                
                # Append to the final dataframe list
                final_df_list.append(csv_df)

# Process files from both directories
process_directory(dir_t20s)
process_directory(dir_it20s)

# Combine all individual DataFrames into one
final_df = pd.concat(final_df_list, ignore_index=True)

# Output the final DataFrame
print(final_df.head())


  Inning_Number Over.Ball Batting_Team      Batsman   Non_Striker     Bowler  \
0             1       0.1    Sri Lanka  GWHM Perera  AC Jayangani  Anam Amin   
1             1       0.2    Sri Lanka  GWHM Perera  AC Jayangani  Anam Amin   
2             1       0.3    Sri Lanka  GWHM Perera  AC Jayangani  Anam Amin   
3             1       0.4    Sri Lanka  GWHM Perera  AC Jayangani  Anam Amin   
4             1       0.5    Sri Lanka     H Madavi  AC Jayangani  Anam Amin   

  Runs_off_Bat Extras Wides No_Balls  ... Dissmissed_Player_When_Wicket  \
0            0      0                 ...                                 
1            4      0                 ...                                 
2            0      0                 ...                                 
3            0      0                 ...                   GWHM Perera   
4            0      0                 ...                                 

      city  gender        date toss_winner toss_decision  \
0  Karac

In [2]:
final_df_list[1]

,Inning_Number,Over.Ball,Batting_Team,Batsman,Non_Striker,Bowler,Runs_off_Bat,Extras,Wides,No_Balls,...,Dissmissed_Player_When_Wicket,city,gender,date,toss_winner,toss_decision,venue,Team1,Team2,match_id
0,1,0.1,Indonesia,NPAN Sakarini,NLKWR Dewi,LAR Bommareddy,0,0,,,...,,Bali,female,2024/07/05,Indonesia,bat,Udayana Cricket Ground,Indonesia,Singapore,1440447.csv
1,1,0.2,Indonesia,NPAN Sakarini,NLKWR Dewi,LAR Bommareddy,0,0,,,...,,Bali,female,2024/07/05,Indonesia,bat,Udayana Cricket Ground,Indonesia,Singapore,1440447.csv
2,1,0.3,Indonesia,NPAN Sakarini,NLKWR Dewi,LAR Bommareddy,1,0,,,...,,Bali,female,2024/07/05,Indonesia,bat,Udayana Cricket Ground,Indonesia,Singapore,1440447.csv
3,1,0.4,Indonesia,NLKWR Dewi,NPAN Sakarini,LAR Bommareddy,0,1,1,,...,,Bali,female,2024/07/05,Indonesia,bat,Udayana Cricket Ground,Indonesia,Singapore,1440447.csv
4,1,0.5,Indonesia,NLKWR Dewi,NPAN Sakarini,LAR Bommareddy,2,0,,,...,,Bali,female,2024/07/05,Indonesia,bat,Udayana Cricket Ground,Indonesia,Singapore,1440447.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
258,2,18.5,Singapore,PSR Gurusinghe,Damini Ramesh,RD Pangestuti,0,1,1,,...,,Bali,female,2024/07/05,Indonesia,bat,Udayana Cricket Ground,Indonesia,Singapore,1440447.csv
259,2,18.6,Singapore,PSR Gurusinghe,Damini Ramesh,RD Pangestuti,2,0,,,...,,Bali,female,2024/07/05,Indonesia,bat,Udayana Cricket Ground,Indonesia,Singapore,1440447.csv
260,2,18.7,Singapore,PSR Gurusinghe,Damini Ramesh,RD Pangestuti,0,0,,,...,PSR Gurusinghe,Bali,female,2024/07/05,Indonesia,bat,Udayana Cricket Ground,Indonesia,Singapore,1440447.csv
261,2,19.1,Singapore,Damini Ramesh,JS Pooranakaran,NLKWR Dewi,0,0,,,...,,Bali,female,2024/07/05,Indonesia,bat,Udayana Cricket Ground,Indonesia,Singapore,1440447.csv


In [3]:
final_df.columns

Index(['Inning_Number', 'Over.Ball', 'Batting_Team', 'Batsman', 'Non_Striker',
       'Bowler', 'Runs_off_Bat', 'Extras', 'Wides', 'No_Balls', 'Byes',
       'Leg_Byes', 'Penalty', 'Kind_of_wicket_If_Any',
       'Dissmissed_Player_When_Wicket', 'city', 'gender', 'date',
       'toss_winner', 'toss_decision', 'venue', 'Team1', 'Team2', 'match_id'],
      dtype='object')

In [4]:
final_df=pd.concat(final_df_list)

In [5]:
final_df

,Inning_Number,Over.Ball,Batting_Team,Batsman,Non_Striker,Bowler,Runs_off_Bat,Extras,Wides,No_Balls,...,Dissmissed_Player_When_Wicket,city,gender,date,toss_winner,toss_decision,venue,Team1,Team2,match_id
0,1,0.1,Sri Lanka,GWHM Perera,AC Jayangani,Anam Amin,0,0,,,...,,Karachi,female,2022/05/24,Sri Lanka,bat,"Southend Club Cricket Stadium, Karachi",Sri Lanka,Pakistan,1310981.csv
1,1,0.2,Sri Lanka,GWHM Perera,AC Jayangani,Anam Amin,4,0,,,...,,Karachi,female,2022/05/24,Sri Lanka,bat,"Southend Club Cricket Stadium, Karachi",Sri Lanka,Pakistan,1310981.csv
2,1,0.3,Sri Lanka,GWHM Perera,AC Jayangani,Anam Amin,0,0,,,...,,Karachi,female,2022/05/24,Sri Lanka,bat,"Southend Club Cricket Stadium, Karachi",Sri Lanka,Pakistan,1310981.csv
3,1,0.4,Sri Lanka,GWHM Perera,AC Jayangani,Anam Amin,0,0,,,...,GWHM Perera,Karachi,female,2022/05/24,Sri Lanka,bat,"Southend Club Cricket Stadium, Karachi",Sri Lanka,Pakistan,1310981.csv
4,1,0.5,Sri Lanka,H Madavi,AC Jayangani,Anam Amin,0,0,,,...,,Karachi,female,2022/05/24,Sri Lanka,bat,"Southend Club Cricket Stadium, Karachi",Sri Lanka,Pakistan,1310981.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196,2,11.7,Botswana,W Tajbhay,I Master,Riaz Suleman,0,0,,,...,,Gaborone,male,2018/11/02,Lesotho,bat,Botswana Cricket Association Oval 2,Lesotho,Botswana,1163064.csv
197,2,12.1,Botswana,I Master,W Tajbhay,T Mohapi,1,0,,,...,,Gaborone,male,2018/11/02,Lesotho,bat,Botswana Cricket Association Oval 2,Lesotho,Botswana,1163064.csv
198,2,12.2,Botswana,W Tajbhay,I Master,T Mohapi,1,0,,,...,,Gaborone,male,2018/11/02,Lesotho,bat,Botswana Cricket Association Oval 2,Lesotho,Botswana,1163064.csv
199,2,12.3,Botswana,I Master,W Tajbhay,T Mohapi,2,0,,,...,,Gaborone,male,2018/11/02,Lesotho,bat,Botswana Cricket Association Oval 2,Lesotho,Botswana,1163064.csv


In [6]:
final_df.to_csv('final_combined_Bharath.csv')

In [7]:
unique_match_id_count = final_df['match_id'].nunique()
print(unique_match_id_count)

3935


In [8]:
final_df['Batting_Team'].nunique()

104

In [9]:
Mainteams = ['Australia', 'South Africa', 'India', 'Pakistan', 'Bangladesh', 
         'Sri Lanka', 'West Indies', 'Afghanistan', 'New Zealand', 'England']



final_male_df = final_df[(final_df['gender'] == 'male')]

filtered_df = final_male_df[
    (final_male_df['Team1'].isin(Mainteams))  & (final_male_df['Team2'].isin(Mainteams))
]



In [10]:
print(final_male_df['Batting_Team'].nunique())
print(filtered_df['Batting_Team'].nunique())


102
10


In [11]:
filtered_df.to_csv('final_mainteams_combined_Bharath.csv')

In [12]:
unique_match_id_count = filtered_df['match_id'].nunique()
print(unique_match_id_count)

837


In [13]:
filtered_df['Batting_Team'].unique()

array(['Australia', 'Pakistan', 'West Indies', 'England', 'India',
       'South Africa', 'Sri Lanka', 'Bangladesh', 'New Zealand',
       'Afghanistan'], dtype=object)

In [14]:
unique_players_count = filtered_df['Bowler'].nunique()
print(unique_players_count)

654


In [15]:
min_date = filtered_df['date'].min()
print("Minimum date:", min_date)

# Find the maximum date
max_date = filtered_df['date'].max()
print("Maximum date:", max_date)

Minimum date: 2005/02/17
Maximum date: 2024/06/29


In [16]:
pip install pandas openpyxl

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [35]:
player_data = pd.read_excel('player_info_updated.xlsx')
print(player_data.head())
print(player_data.columns)

  Player Name        Bowling type    Batting type         Playing role  \
0     V Kohli    Right arm Medium  Right hand Bat     Top order Batter   
1   RG Sharma  Right arm Offbreak  Right hand Bat     Top order Batter   
2  Babar Azam  Right arm Offbreak  Right hand Bat               Batter   
3   DA Warner            Legbreak   Left hand Bat       Opening Batter   
4  JC Buttler                 NaN  Right hand Bat  Wicketkeeper Batter   

  Batting hand Bowling hand Bowling typeupdated Coreskill  
0        Right        Right              Medium    Batter  
1        Right        Right            Off spin    Batter  
2        Right        Right            Off spin    Batter  
3         Left         Left            Leg spin    Batter  
4        Right        Right                 NaN    Batter  
Index(['Player Name', 'Bowling type', 'Batting type', 'Playing role',
       'Batting hand', 'Bowling hand', 'Bowling typeupdated', 'Coreskill'],
      dtype='object')


In [36]:
player_data.head()

,Player Name,Bowling type,Batting type,Playing role,Batting hand,Bowling hand,Bowling typeupdated,Coreskill
0,V Kohli,Right arm Medium,Right hand Bat,Top order Batter,Right,Right,Medium,Batter
1,RG Sharma,Right arm Offbreak,Right hand Bat,Top order Batter,Right,Right,Off spin,Batter
2,Babar Azam,Right arm Offbreak,Right hand Bat,Batter,Right,Right,Off spin,Batter
3,DA Warner,Legbreak,Left hand Bat,Opening Batter,Left,Left,Leg spin,Batter
4,JC Buttler,NaN,Right hand Bat,Wicketkeeper Batter,Right,Right,NaN,Batter


In [37]:
player_data_batter = player_data[(player_data['Coreskill'].isin(['Batter', 'Bowler and Batter', 'Batter and Bowler']))]
player_data_bowler = player_data[(player_data['Coreskill'].isin(['Bowler', 'Bowler and Batter', 'Batter and Bowler']))]
unique_playersbatter_count = player_data_batter['Player Name'].nunique()
unique_playersbowler_count = player_data_bowler['Player Name'].nunique()
print(unique_playersbatter_count)
print(unique_playersbowler_count)

598
590


In [38]:

# Select only the necessary columns for Batsman and Bowler from the first dataset
batsman_info = player_data[['Player Name', 'Coreskill', 'Batting hand']].copy()
bowler_info = player_data[['Player Name', 'Coreskill', 'Bowling hand', 'Bowling typeupdated']].copy()

# Rename columns to avoid conflicts during the merge
batsman_info.columns = ['Batsman', 'Batsman_coreskill', 'Batsman_hand']
bowler_info.columns = ['Bowler', 'Bowler_coreskill', 'Bowling_hand', 'Bowling_style']

# Merge the second dataset with the batsman and bowler information
merged_df = filtered_df.merge(batsman_info, on='Batsman', how='left')
merged_df = merged_df.merge(bowler_info, on='Bowler', how='left')

# Save the updated dataset
#merged_df.to_csv('updated_second_dataset.csv', index=False)

# Print the updated DataFrame to check:
print(merged_df.head())


  Inning_Number Over.Ball Batting_Team    Batsman Non_Striker         Bowler  \
0             1       0.1    Australia  DJM Short    AJ Finch  Mohammad Amir   
1             1       0.2    Australia  DJM Short    AJ Finch  Mohammad Amir   
2             1       0.3    Australia  DJM Short    AJ Finch  Mohammad Amir   
3             1       0.4    Australia  DJM Short    AJ Finch  Mohammad Amir   
4             1       0.5    Australia  DJM Short    AJ Finch  Mohammad Amir   

  Runs_off_Bat Extras Wides No_Balls  ... toss_decision               venue  \
0            4      0                 ...           bat  Harare Sports Club   
1            0      0                 ...           bat  Harare Sports Club   
2            4      0                 ...           bat  Harare Sports Club   
3            0      0                 ...           bat  Harare Sports Club   
4            0      0                 ...           bat  Harare Sports Club   

       Team1     Team2     match_id Batsman_

In [39]:
merged_df.head()

,Inning_Number,Over.Ball,Batting_Team,Batsman,Non_Striker,Bowler,Runs_off_Bat,Extras,Wides,No_Balls,...,toss_decision,venue,Team1,Team2,match_id,Batsman_coreskill,Batsman_hand,Bowler_coreskill,Bowling_hand,Bowling_style
0,1,0.1,Australia,DJM Short,AJ Finch,Mohammad Amir,4,0,,,...,bat,Harare Sports Club,Australia,Pakistan,1142919.csv,Batter,Left,Bowler,Left,Fast
1,1,0.2,Australia,DJM Short,AJ Finch,Mohammad Amir,0,0,,,...,bat,Harare Sports Club,Australia,Pakistan,1142919.csv,Batter,Left,Bowler,Left,Fast
2,1,0.3,Australia,DJM Short,AJ Finch,Mohammad Amir,4,0,,,...,bat,Harare Sports Club,Australia,Pakistan,1142919.csv,Batter,Left,Bowler,Left,Fast
3,1,0.4,Australia,DJM Short,AJ Finch,Mohammad Amir,0,0,,,...,bat,Harare Sports Club,Australia,Pakistan,1142919.csv,Batter,Left,Bowler,Left,Fast
4,1,0.5,Australia,DJM Short,AJ Finch,Mohammad Amir,0,0,,,...,bat,Harare Sports Club,Australia,Pakistan,1142919.csv,Batter,Left,Bowler,Left,Fast


In [31]:
merged_df.to_csv('updated_combined_dataset.csv', index=False)

In [40]:
# Check which columns have missing values and the count of missing values per column
missing_values_per_column = merged_df.isnull().sum()

# Output columns with missing values
missing_columns = missing_values_per_column[missing_values_per_column > 0]
print(f"Columns with missing values:\n{missing_columns}")

Columns with missing values:
Batsman_coreskill     88
Batsman_hand          88
Bowler_coreskill     375
Bowling_hand         375
Bowling_style        377
dtype: int64


In [69]:
# Identify rows where 'Batsman_coreskill' or 'Batsman_hand' are missing
missing_batsman_info = merged_df[merged_df['Batsman_coreskill'].isnull() | merged_df['Batsman_hand'].isnull()]

# Get unique batsmen with missing information
unique_missing_batsmen = missing_batsman_info['Batsman'].unique()

# Display the unique batsmen with missing information
print(f"Unique Batsmen with missing Batsman_coreskill or Batsman_hand:\n{unique_missing_batsmen}")

# Check if these batsmen are present in the 'Player Name' column of the first dataset
missing_batsmen_not_in_first = [batsman for batsman in unique_missing_batsmen if batsman not in player_data['Player Name'].values]

# Get the teams for the batsmen with missing information
teams_of_missing_batsmen = {}

for batsman in unique_missing_batsmen:
    if batsman in merged_df['Batsman'].values:
        # Get the unique teams associated with this batsman
        teams = merged_df.loc[merged_df['Batsman'] == batsman, 'Batting_Team'].unique()
        teams_of_missing_batsmen[batsman] = teams

# Display the results
if missing_batsmen_not_in_first:
    print(f"These batsmen are missing in the first dataset 'Player Name' column:\n{missing_batsmen_not_in_first}")
else:
    print("All batsmen with missing Batsman_coreskill or Batsman_hand are present in the first dataset.")

print("\nTeams of batsmen with missing Batsman_coreskill or Batsman_hand:")
for batsman, teams in teams_of_missing_batsmen.items():
    print(f"{batsman}: {', '.join(teams)}")


Unique Batsmen with missing Batsman_coreskill or Batsman_hand:
['Mohammad Nawaz (3)' 'Aminul Islam' 'Navdeep Saini']
These batsmen are missing in the first dataset 'Player Name' column:
['Mohammad Nawaz (3)', 'Aminul Islam', 'Navdeep Saini']

Teams of batsmen with missing Batsman_coreskill or Batsman_hand:
Mohammad Nawaz (3): Pakistan
Aminul Islam: Bangladesh
Navdeep Saini: India


In [70]:
cleaned_df = merged_df.dropna(subset=['Batsman_coreskill', 'Batsman_hand'])

In [71]:
# Identify rows where 'Bowler_coreskill' or 'Bowling_hand' or 'Bowling_style' are missing
missing_bowling_info = cleaned_df[merged_df['Bowler_coreskill'].isnull() | merged_df['Bowling_hand'].isnull() | merged_df['Bowling_style'].isnull()]
# Get unique bowler with missing information
unique_missing_bowlers = missing_bowling_info['Bowler'].unique()

# Display the unique bowler with missing information
print(f"Unique Bowler with missing Bowler_coreskill or Bowler_hand or Bowling_style:\n{unique_missing_bowlers}")

# Check if these bowlers are present in the 'Player Name' column of the first dataset
missing_bowlers_not_in_first = [bowler for bowler in unique_missing_bowlers if bowler not in player_data['Player Name'].values]

# Get the teams for the bowlers with missing information
teams_of_missing_bowlers = {}

for bowler in unique_missing_bowlers:
    if bowler in merged_df['Bowler'].values:
        # Get the unique teams associated with this bowler
        teams = merged_df.loc[merged_df['Bowler'] == bowler, 'Team2'].unique()
        teams_of_missing_bowlers[bowler] = teams

# Display the results
if missing_bowlers_not_in_first:
    print(f"These bowlers are missing in the first dataset 'Player Name' column:\n{missing_bowlers_not_in_first}")
else:
    print("All bowlers with missing Bowler_coreskill or Bowler_hand are present in the first dataset.")

print("\nTeams of batsmen with missing Bowler_coreskill or Bowler_hand:")
for bowler, teams in teams_of_missing_bowlers.items():
    print(f"{bowler}: {', '.join(teams)}")



Unique Bowler with missing Bowler_coreskill or Bowler_hand or Bowling_style:
['Mohammad Nawaz (3)' 'Mahmudul Hasan Joy' 'Aminul Islam']
These bowlers are missing in the first dataset 'Player Name' column:
['Mohammad Nawaz (3)', 'Aminul Islam']

Teams of batsmen with missing Bowler_coreskill or Bowler_hand:
Mohammad Nawaz (3): West Indies, Pakistan, Sri Lanka, South Africa, New Zealand, Australia
Mahmudul Hasan Joy: India
Aminul Islam: Pakistan


/var/folders/v3/g1h3k6911lvdjsq_7nzv6s700000gn/T/ipykernel_40263/711722241.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  missing_bowling_info = cleaned_df[merged_df['Bowler_coreskill'].isnull() | merged_df['Bowling_hand'].isnull() | merged_df['Bowling_style'].isnull()]


In [72]:
cleaned_df = cleaned_df.dropna(subset=['Bowler_coreskill', 'Bowling_hand', 'Bowling_style'])

In [73]:
cleaned_df.head()

,Inning_Number,Over.Ball,Batting_Team,Batsman,Non_Striker,Bowler,Runs_off_Bat,Extras,Wides,No_Balls,...,toss_decision,venue,Team1,Team2,match_id,Batsman_coreskill,Batsman_hand,Bowler_coreskill,Bowling_hand,Bowling_style
0,1,0.1,Australia,DJM Short,AJ Finch,Mohammad Amir,4,0,,,...,bat,Harare Sports Club,Australia,Pakistan,1142919.csv,Batter,Left,Bowler,Left,Fast
1,1,0.2,Australia,DJM Short,AJ Finch,Mohammad Amir,0,0,,,...,bat,Harare Sports Club,Australia,Pakistan,1142919.csv,Batter,Left,Bowler,Left,Fast
2,1,0.3,Australia,DJM Short,AJ Finch,Mohammad Amir,4,0,,,...,bat,Harare Sports Club,Australia,Pakistan,1142919.csv,Batter,Left,Bowler,Left,Fast
3,1,0.4,Australia,DJM Short,AJ Finch,Mohammad Amir,0,0,,,...,bat,Harare Sports Club,Australia,Pakistan,1142919.csv,Batter,Left,Bowler,Left,Fast
4,1,0.5,Australia,DJM Short,AJ Finch,Mohammad Amir,0,0,,,...,bat,Harare Sports Club,Australia,Pakistan,1142919.csv,Batter,Left,Bowler,Left,Fast


In [74]:
# Check which columns have missing values and the count of missing values per column
missing_values_per_column = cleaned_df.isnull().sum()

# Output columns with missing values
missing_columns = missing_values_per_column[missing_values_per_column > 0]
print(f"Columns with missing values:\n{missing_columns}")

Columns with missing values:
Series([], dtype: int64)


In [76]:
cleaned_df['Over']=cleaned_df['Over.Ball'].apply(lambda x:str(x).split('.')[0])
cleaned_df['Ball']=cleaned_df['Over.Ball'].apply(lambda x:str(x).split('.')[1])
cleaned_df['Over']=cleaned_df['Over'].astype(int)
cleaned_df['Over']  = cleaned_df['Over'] + 1
cleaned_df['Ball']=cleaned_df['Ball'].astype(int)

In [205]:


# Trim characters after the comma in the 'Venue' column
cleaned_df['venue'] = cleaned_df['venue'].str.split(',').str[0]

# Create a dictionary for mapping incorrect names to correct ones
corrections = {
    "Daren Sammy National Cricket Stadium": "Darren Sammy National Cricket Stadium",
    "Eden Park": "Eden Gardens",
    "Kennington Oval": "Kensington Oval",
    "M.Chinnaswamy Stadium": "MA Chidambaram Stadium",
    "Rajiv Gandhi International Stadium": "Rajiv Gandhi International Cricket Stadium",
    "R Premadasa Stadium": "R.Premadasa Stadium",
    "Punjab Cricket Association Stadium": "Punjab Cricket Association IS Bindra Stadium",
    "National Stadium": "National Cricket Stadium"
}

# Replace incorrect names with correct ones
cleaned_df['venue'] = cleaned_df['venue'].replace(corrections)

# Display the first few rows to verify
print(cleaned_df.head())

        Inning_Number  Over.Ball Batting_Team     Batsman Non_Striker  \
129262              2       10.6  West Indies  A Athanaze   KR Mayers   
129268              2       11.6  West Indies  A Athanaze   KR Mayers   
129269              2       11.7  West Indies  A Athanaze   KR Mayers   
129271              2       12.2  West Indies  A Athanaze   KR Mayers   
129276              2       12.7  West Indies  A Athanaze   KR Mayers   

              Bowler  Runs_off_Bat  Extras  Wides  No_Balls  ...        Team2  \
129262     G Coetzee             0       0    NaN       NaN  ...  West Indies   
129268  RR Hendricks             0       0    NaN       NaN  ...  West Indies   
129269  RR Hendricks             2       0    NaN       NaN  ...  West Indies   
129271     G Coetzee             1       0    NaN       NaN  ...  West Indies   
129276     G Coetzee             0       1    1.0       NaN  ...  West Indies   

           match_id  Batsman_coreskill Batsman_hand Bowler_coreskill  \
12

In [206]:
cleaned_df.to_csv('cleaned_combined_dataset.csv', index=False)

In [220]:
import pandas as pd

# Load the dataset
df = pd.read_csv('cleaned_combined_dataset.csv')

# Convert Runs_off_Bat to numeric, handling errors
df['Runs_off_Bat'] = pd.to_numeric(df['Runs_off_Bat'], errors='coerce')

# Assume each row represents a ball faced
df['Balls faced'] = 1

# Determine the Bowling_Team team
df['Bowling_Team'] = df.apply(lambda x: x['Team2'] if x['Batting_Team'] == x['Team1'] else x['Team1'], axis=1)

# Create the 'dismissal' column
df['dismissal'] = df.apply(lambda x: 1 if x['Batsman'] == x['Dissmissed_Player_When_Wicket'] else 0, axis=1)


# Initialize a new DataFrame to store results
cumulative_df = pd.DataFrame()

# Group by Batsman and match_id to calculate cumulative statistics
grouped = df.groupby(['Batsman', 'match_id'])

# Aggregating fields
cumulative_df['Innings'] = grouped['Inning_Number'].first()  # Taking the first Inning_Number for each group
cumulative_df['Runs scored'] = grouped['Runs_off_Bat'].sum()
cumulative_df['Balls faced'] = grouped['Balls faced'].sum()
cumulative_df['Batting Strike Rate (SR)'] = (cumulative_df['Runs scored'] / cumulative_df['Balls faced']) * 100
cumulative_df['No. of Sixes'] = grouped.apply(lambda x: (x['Runs_off_Bat'] == 6).sum())
cumulative_df['No. of Fours'] = grouped.apply(lambda x: (x['Runs_off_Bat'] == 4).sum())
cumulative_df['dismissal'] = grouped['dismissal'].max()  # Take the maximum dismissal value (1 or 0) per match


# Adding Batting hand, Venue/Ground, Bowling_Team, and Toss decision
cumulative_df['Batting hand'] = grouped['Batsman_hand'].first()
cumulative_df['Ground'] = grouped['venue'].first()
cumulative_df['Bowling_Team'] = grouped['Bowling_Team'].first()
cumulative_df['Toss'] = grouped['toss_decision'].first()
cumulative_df['Coreskill'] = grouped['Batsman_coreskill'].first()
cumulative_df['Match_date'] = grouped['date'].first()

# Counting bowlers faced by type
cumulative_df['No of Right hand bowlers_faced(RHBo)'] = grouped.apply(lambda x: (x['Bowling_hand'] == 'Right').sum())
cumulative_df['No of left hand bowlers_faced(LHBo)'] = grouped.apply(lambda x: (x['Bowling_hand'] == 'Left').sum())
cumulative_df['No of Off Spinners_faced(OffspinBo)'] = grouped.apply(lambda x: (x['Bowling_style'].str.contains('Off spin')).sum())
cumulative_df['No of leg Spinners_faced(LegspinBo)'] = grouped.apply(lambda x: (x['Bowling_style'].str.contains('Leg spin')).sum())
cumulative_df['No of Fast pace  Bowlers_faced(FastBo)'] = grouped.apply(lambda x: (x['Bowling_style'].str.contains('Fast')).sum())
cumulative_df['No of Medium pace bowlers_faced(MediumBo)'] = grouped.apply(lambda x: (x['Bowling_style'].str.contains('Medium')).sum())
cumulative_df['No of Slow pace bowlers_faced(SlowBo)'] = grouped.apply(lambda x: (x['Bowling_style'].str.contains('Slow')).sum())

# Reset the index to flatten the DataFrame
cumulative_df.reset_index(inplace=True)

# Save the cumulative results to a new CSV file
cumulative_df.to_csv('cumulative_batsman_stats_dismissals.csv', index=False)

# Display the first few rows of the new DataFrame to verify
print(cumulative_df.head())


/var/folders/v3/g1h3k6911lvdjsq_7nzv6s700000gn/T/ipykernel_40263/3994297736.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cumulative_df['No. of Sixes'] = grouped.apply(lambda x: (x['Runs_off_Bat'] == 6).sum())
/var/folders/v3/g1h3k6911lvdjsq_7nzv6s700000gn/T/ipykernel_40263/3994297736.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cumulative_df['No. of Fours'] = grouped.apply(lambda x: (x['Runs

       Batsman     match_id  Innings  Runs scored  Balls faced  \
0   A Athanaze  1433364.csv        2            6            8   
1  A Dananjaya  1122729.csv        2            7            8   
2  A Dananjaya  1122730.csv        2            5            4   
3  A Dananjaya  1122731.csv        1           11            7   
4  A Dananjaya  1133820.csv        1            5           11   

   Batting Strike Rate (SR)  No. of Sixes  No. of Fours  dismissal  \
0                 75.000000             0             0          0   
1                 87.500000             0             0          1   
2                125.000000             0             1          1   
3                157.142857             0             2          0   
4                 45.454545             0             0          1   

  Batting hand  ...   Toss          Coreskill  Match_date  \
0         Left  ...    bat             Batter  2024/05/26   
1         Left  ...  field  Batter and Bowler  2017/12/20   

/var/folders/v3/g1h3k6911lvdjsq_7nzv6s700000gn/T/ipykernel_40263/3994297736.py:50: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cumulative_df['No of Slow pace bowlers_faced(SlowBo)'] = grouped.apply(lambda x: (x['Bowling_style'].str.contains('Slow')).sum())


In [103]:
import pandas as pd

# Load the cleaned combined dataset
df = pd.read_csv('cleaned_combined_dataset.csv')

# Convert necessary columns to the appropriate data types for calculations
df['Runs_off_Bat'] = pd.to_numeric(df['Runs_off_Bat'], errors='coerce')
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Add 'Balls faced' column assuming each row represents a ball faced
df['Balls faced'] = 1

# Sort data by batsman, date, and match_id for cumulative calculations
df = df.sort_values(by=['Batsman', 'date', 'match_id']).reset_index(drop=True)

# Initialize a new DataFrame to store cumulative stats up to each match
cumulative_df = pd.DataFrame()

# Group by Batsman and match_id to calculate cumulative statistics up to each match
grouped = df.groupby(['Batsman', 'match_id'])

# Loop over each batsman to calculate cumulative stats up to each match
cumulative_data = []  # List to store cumulative stats data before converting to DataFrame

for (batsman, match_id), group in grouped:
    # Subset data to include all matches before the current match_id
    batsman_data = df[(df['Batsman'] == batsman) & (df['date'] <= group['date'].iloc[0])]
    
    # Exclude the current match data to get only the previous match stats
    previous_matches = batsman_data[batsman_data['match_id'] != match_id]
    
    # Calculate cumulative stats for matches played before the current match
    cumulative_runs = previous_matches['Runs_off_Bat'].sum()
    cumulative_balls = previous_matches['Balls faced'].sum()
    cumulative_innings = previous_matches['match_id'].nunique()
    
    cumulative_sixes = (previous_matches['Runs_off_Bat'] == 6).sum()
    cumulative_fours = (previous_matches['Runs_off_Bat'] == 4).sum()
    cumulative_highest_score = previous_matches.groupby('match_id')['Runs_off_Bat'].sum().max()
    
    dismissals = previous_matches['Kind_of_wicket_If_Any'].notnull().sum()
    cumulative_average = cumulative_runs / dismissals if dismissals > 0 else pd.NA
    
    cumulative_sr = (cumulative_runs / cumulative_balls * 100) if cumulative_balls > 0 else pd.NA
    
    # Batting Strike Rate against different types of bowlers
    cumulative_sr_rhbo = (previous_matches.loc[previous_matches['Bowling_hand'] == 'Right', 'Runs_off_Bat'].sum() /
                          previous_matches.loc[previous_matches['Bowling_hand'] == 'Right', 'Balls faced'].sum() * 100) if previous_matches.loc[previous_matches['Bowling_hand'] == 'Right', 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_lhbo = (previous_matches.loc[previous_matches['Bowling_hand'] == 'Left', 'Runs_off_Bat'].sum() /
                          previous_matches.loc[previous_matches['Bowling_hand'] == 'Left', 'Balls faced'].sum() * 100) if previous_matches.loc[previous_matches['Bowling_hand'] == 'Left', 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_offspin = (previous_matches.loc[previous_matches['Bowling_style'].str.contains('Off spin', na=False), 'Runs_off_Bat'].sum() /
                             previous_matches.loc[previous_matches['Bowling_style'].str.contains('Off spin', na=False), 'Balls faced'].sum() * 100) if previous_matches.loc[previous_matches['Bowling_style'].str.contains('Off spin', na=False), 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_legspin = (previous_matches.loc[previous_matches['Bowling_style'].str.contains('Leg spin', na=False), 'Runs_off_Bat'].sum() /
                             previous_matches.loc[previous_matches['Bowling_style'].str.contains('Leg spin', na=False), 'Balls faced'].sum() * 100) if previous_matches.loc[previous_matches['Bowling_style'].str.contains('Leg spin', na=False), 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_fastbo = (previous_matches.loc[previous_matches['Bowling_style'].str.contains('Fast', na=False), 'Runs_off_Bat'].sum() /
                            previous_matches.loc[previous_matches['Bowling_style'].str.contains('Fast', na=False), 'Balls faced'].sum() * 100) if previous_matches.loc[previous_matches['Bowling_style'].str.contains('Fast', na=False), 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_mediumbo = (previous_matches.loc[previous_matches['Bowling_style'].str.contains('Medium', na=False), 'Runs_off_Bat'].sum() /
                              previous_matches.loc[previous_matches['Bowling_style'].str.contains('Medium', na=False), 'Balls faced'].sum() * 100) if previous_matches.loc[previous_matches['Bowling_style'].str.contains('Medium', na=False), 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_slowbo = (previous_matches.loc[previous_matches['Bowling_style'].str.contains('Slow', na=False), 'Runs_off_Bat'].sum() /
                            previous_matches.loc[previous_matches['Bowling_style'].str.contains('Slow', na=False), 'Balls faced'].sum() * 100) if previous_matches.loc[previous_matches['Bowling_style'].str.contains('Slow', na=False), 'Balls faced'].sum() > 0 else pd.NA
    
    # Store cumulative data in a dictionary and append to list
    cumulative_data.append({
        'Batsman': batsman,
        'match_id': match_id,
        'Innings_Career': cumulative_innings,
        'Runs scored_Career': cumulative_runs,
        'Balls faced_Career': cumulative_balls,
        'Batting Average_Career': cumulative_average,
        'Batting Strike Rate_Career': cumulative_sr,
        'Centuries_Career': (previous_matches.groupby('match_id')['Runs_off_Bat'].sum() >= 100).sum(),
        'Fifties_Career': (previous_matches.groupby('match_id')['Runs_off_Bat'].sum().between(50, 99)).sum(),
        'Sixes_Career': cumulative_sixes,
        'Fours_Career': cumulative_fours,
        'Highest Score_Career': cumulative_highest_score,
        'Batting Strike Rate_Career_RHBo': cumulative_sr_rhbo,
        'Batting Strike Rate_Career_LHBo': cumulative_sr_lhbo,
        'Batting Strike Rate_Career_OffSpin': cumulative_sr_offspin,
        'Batting Strike Rate_Career_LegSpin': cumulative_sr_legspin,
        'Batting Strike Rate_Career_FastBo': cumulative_sr_fastbo,
        'Batting Strike Rate_Career_MediumBo': cumulative_sr_mediumbo,
        'Batting Strike Rate_Career_SlowBo': cumulative_sr_slowbo
    })

# Convert the list of dictionaries to a DataFrame
cumulative_df = pd.DataFrame(cumulative_data)

# Save the updated cumulative stats back to CSV
cumulative_df.to_csv('cumulative_batsman_stats2.csv', index=False)

# Display the first few rows of the cumulative_df to verify
print(cumulative_df.head())


       Batsman     match_id  Innings_Career  Runs scored_Career  \
0   A Athanaze  1433364.csv               0                   0   
1  A Dananjaya  1122729.csv               0                   0   
2  A Dananjaya  1122730.csv               1                   7   
3  A Dananjaya  1122731.csv               2                  12   
4  A Dananjaya  1133820.csv               3                  23   

   Balls faced_Career Batting Average_Career Batting Strike Rate_Career  \
0                   0                   <NA>                       <NA>   
1                   0                   <NA>                       <NA>   
2                   8                    7.0                       87.5   
3                  12                    6.0                      100.0   
4                  19                   11.5                 121.052632   

   Centuries_Career  Fifties_Career  Sixes_Career  Fours_Career  \
0                 0               0             0             0   
1           

In [208]:
import pandas as pd

# Load the cleaned combined dataset
df = pd.read_csv('cleaned_combined_dataset.csv')

# Convert necessary columns to the appropriate data types for calculations
df['Runs_off_Bat'] = pd.to_numeric(df['Runs_off_Bat'], errors='coerce')
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Add 'Balls faced' column assuming each row represents a ball faced
df['Balls faced'] = 1

# Sort data by batsman, venue, date, and match_id for cumulative calculations
df = df.sort_values(by=['Batsman', 'venue', 'date', 'match_id']).reset_index(drop=True)

# Initialize a new DataFrame to store cumulative stats up to each match
cumulative_df_ground = pd.DataFrame()

# Group by Batsman, venue, and match_id to calculate cumulative statistics up to each match
grouped = df.groupby(['Batsman', 'venue', 'match_id'])

# Loop over each batsman and ground to calculate cumulative stats up to each match
cumulative_data_ground = []  # List to store cumulative stats data before converting to DataFrame

for (batsman, venue, match_id), group in grouped:
    # Subset data to include all matches before the current match_id on the same ground
    batsman_ground_data = df[(df['Batsman'] == batsman) & (df['venue'] == venue) & (df['date'] <= group['date'].iloc[0])]
    
    # Exclude the current match data to get only the previous match stats on the same ground
    previous_ground_matches = batsman_ground_data[batsman_ground_data['match_id'] != match_id]
    
    # Calculate cumulative stats for matches played on the same ground before the current match
    cumulative_runs_ground = previous_ground_matches['Runs_off_Bat'].sum()
    cumulative_balls_ground = previous_ground_matches['Balls faced'].sum()
    cumulative_innings_ground = previous_ground_matches['match_id'].nunique()
    
    cumulative_sixes_ground = (previous_ground_matches['Runs_off_Bat'] == 6).sum()
    cumulative_fours_ground = (previous_ground_matches['Runs_off_Bat'] == 4).sum()
    cumulative_highest_score_ground = previous_ground_matches.groupby('match_id')['Runs_off_Bat'].sum().max()
    
    dismissals_ground = previous_ground_matches['Kind_of_wicket_If_Any'].notnull().sum()
    cumulative_average_ground = cumulative_runs_ground / dismissals_ground if dismissals_ground > 0 else pd.NA
    
    cumulative_sr_ground = (cumulative_runs_ground / cumulative_balls_ground * 100) if cumulative_balls_ground > 0 else pd.NA
    
    # Batting Strike Rate against different types of bowlers on the same ground
    cumulative_sr_rhbo_ground = (previous_ground_matches.loc[previous_ground_matches['Bowling_hand'] == 'Right', 'Runs_off_Bat'].sum() /
                                 previous_ground_matches.loc[previous_ground_matches['Bowling_hand'] == 'Right', 'Balls faced'].sum() * 100) if previous_ground_matches.loc[previous_ground_matches['Bowling_hand'] == 'Right', 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_lhbo_ground = (previous_ground_matches.loc[previous_ground_matches['Bowling_hand'] == 'Left', 'Runs_off_Bat'].sum() /
                                 previous_ground_matches.loc[previous_ground_matches['Bowling_hand'] == 'Left', 'Balls faced'].sum() * 100) if previous_ground_matches.loc[previous_ground_matches['Bowling_hand'] == 'Left', 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_offspin_ground = (previous_ground_matches.loc[previous_ground_matches['Bowling_style'].str.contains('Off spin', na=False), 'Runs_off_Bat'].sum() /
                                    previous_ground_matches.loc[previous_ground_matches['Bowling_style'].str.contains('Off spin', na=False), 'Balls faced'].sum() * 100) if previous_ground_matches.loc[previous_ground_matches['Bowling_style'].str.contains('Off spin', na=False), 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_legspin_ground = (previous_ground_matches.loc[previous_ground_matches['Bowling_style'].str.contains('Leg spin', na=False), 'Runs_off_Bat'].sum() /
                                    previous_ground_matches.loc[previous_ground_matches['Bowling_style'].str.contains('Leg spin', na=False), 'Balls faced'].sum() * 100) if previous_ground_matches.loc[previous_ground_matches['Bowling_style'].str.contains('Leg spin', na=False), 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_fastbo_ground = (previous_ground_matches.loc[previous_ground_matches['Bowling_style'].str.contains('Fast', na=False), 'Runs_off_Bat'].sum() /
                                   previous_ground_matches.loc[previous_ground_matches['Bowling_style'].str.contains('Fast', na=False), 'Balls faced'].sum() * 100) if previous_ground_matches.loc[previous_ground_matches['Bowling_style'].str.contains('Fast', na=False), 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_mediumbo_ground = (previous_ground_matches.loc[previous_ground_matches['Bowling_style'].str.contains('Medium', na=False), 'Runs_off_Bat'].sum() /
                                     previous_ground_matches.loc[previous_ground_matches['Bowling_style'].str.contains('Medium', na=False), 'Balls faced'].sum() * 100) if previous_ground_matches.loc[previous_ground_matches['Bowling_style'].str.contains('Medium', na=False), 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_slowbo_ground = (previous_ground_matches.loc[previous_ground_matches['Bowling_style'].str.contains('Slow', na=False), 'Runs_off_Bat'].sum() /
                                   previous_ground_matches.loc[previous_ground_matches['Bowling_style'].str.contains('Slow', na=False), 'Balls faced'].sum() * 100) if previous_ground_matches.loc[previous_ground_matches['Bowling_style'].str.contains('Slow', na=False), 'Balls faced'].sum() > 0 else pd.NA
    
    # Store cumulative data in a dictionary and append to list
    cumulative_data_ground.append({
        'Batsman': batsman,
        'venue': venue,
        'match_id': match_id,
        'No. of Innings_ground': cumulative_innings_ground,
        'Runs scored_ground': cumulative_runs_ground,
        'Balls faced_ground': cumulative_balls_ground,
        'Batting Average_ground': cumulative_average_ground,
        'Batting Strike Rate_ground': cumulative_sr_ground,
        'Centuries_ground': (previous_ground_matches.groupby('match_id')['Runs_off_Bat'].sum() >= 100).sum(),
        'Fifties_ground': (previous_ground_matches.groupby('match_id')['Runs_off_Bat'].sum().between(50, 99)).sum(),
        'Sixes_ground': cumulative_sixes_ground,
        'Fours_ground': cumulative_fours_ground,
        'Highest Score_ground': cumulative_highest_score_ground,
        'Batting Strike Rate_ground_RHBo': cumulative_sr_rhbo_ground,
        'Batting Strike Rate_ground_LHBo': cumulative_sr_lhbo_ground,
        'Batting Strike Rate_ground_OffSpin': cumulative_sr_offspin_ground,
        'Batting Strike Rate_ground_LegSpin': cumulative_sr_legspin_ground,
        'Batting Strike Rate_ground_FastBo': cumulative_sr_fastbo_ground,
        'Batting Strike Rate_ground_MediumBo': cumulative_sr_mediumbo_ground,
        'Batting Strike Rate_ground_SlowBo': cumulative_sr_slowbo_ground
    })

# Convert the list of dictionaries to a DataFrame
cumulative_df_ground = pd.DataFrame(cumulative_data_ground)

# Save the updated cumulative stats back to CSV
cumulative_df_ground.to_csv('cumulative_batsman_stats_ground.csv', index=False)

# Display the first few rows of the cumulative_df to verify
print(cumulative_df_ground.head())


       Batsman                    venue     match_id  No. of Innings_ground  \
0   A Athanaze              Sabina Park  1433364.csv                      0   
1  A Dananjaya         Barabati Stadium  1122729.csv                      0   
2  A Dananjaya  Coolidge Cricket Ground  1252066.csv                      0   
3  A Dananjaya   Holkar Cricket Stadium  1122730.csv                      0   
4  A Dananjaya                 Newlands  1144172.csv                      0   

   Runs scored_ground  Balls faced_ground Batting Average_ground  \
0                   0                   0                   <NA>   
1                   0                   0                   <NA>   
2                   0                   0                   <NA>   
3                   0                   0                   <NA>   
4                   0                   0                   <NA>   

  Batting Strike Rate_ground  Centuries_ground  Fifties_ground  Sixes_ground  \
0                       <NA>        

In [105]:
import pandas as pd

# Load the cleaned combined dataset
df = pd.read_csv('cleaned_combined_dataset.csv')

# Convert necessary columns to the appropriate data types for calculations
df['Runs_off_Bat'] = pd.to_numeric(df['Runs_off_Bat'], errors='coerce')
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Add 'Balls faced' column assuming each row represents a ball faced
df['Balls faced'] = 1

# Determine the Opposition team
df['Opposition'] = df.apply(lambda x: x['Team2'] if x['Batting_Team'] == x['Team1'] else x['Team1'], axis=1)

# Sort data by batsman, opposition, date, and match_id for cumulative calculations
df = df.sort_values(by=['Batsman', 'Opposition', 'date', 'match_id']).reset_index(drop=True)

# Initialize a new DataFrame to store cumulative stats up to each match
cumulative_df_opposition = pd.DataFrame()

# Group by Batsman, Opposition, and match_id to calculate cumulative statistics up to each match
grouped = df.groupby(['Batsman', 'Opposition', 'match_id'])

# Loop over each batsman and opposition to calculate cumulative stats up to each match
cumulative_data_opposition = []  # List to store cumulative stats data before converting to DataFrame

for (batsman, opposition, match_id), group in grouped:
    # Subset data to include all matches before the current match_id against the same opposition
    batsman_opposition_data = df[(df['Batsman'] == batsman) & (df['Opposition'] == opposition) & (df['date'] <= group['date'].iloc[0])]
    
    # Exclude the current match data to get only the previous match stats against the same opposition
    previous_opposition_matches = batsman_opposition_data[batsman_opposition_data['match_id'] != match_id]
    
    # Calculate cumulative stats for matches played against the same opposition before the current match
    cumulative_runs_opposition = previous_opposition_matches['Runs_off_Bat'].sum()
    cumulative_balls_opposition = previous_opposition_matches['Balls faced'].sum()
    cumulative_innings_opposition = previous_opposition_matches['match_id'].nunique()
    
    cumulative_sixes_opposition = (previous_opposition_matches['Runs_off_Bat'] == 6).sum()
    cumulative_fours_opposition = (previous_opposition_matches['Runs_off_Bat'] == 4).sum()
    cumulative_highest_score_opposition = previous_opposition_matches.groupby('match_id')['Runs_off_Bat'].sum().max()
    
    dismissals_opposition = previous_opposition_matches['Kind_of_wicket_If_Any'].notnull().sum()
    cumulative_average_opposition = cumulative_runs_opposition / dismissals_opposition if dismissals_opposition > 0 else pd.NA
    
    cumulative_sr_opposition = (cumulative_runs_opposition / cumulative_balls_opposition * 100) if cumulative_balls_opposition > 0 else pd.NA
    
    # Batting Strike Rate against different types of bowlers against the same opposition
    cumulative_sr_rhbo_opposition = (previous_opposition_matches.loc[previous_opposition_matches['Bowling_hand'] == 'Right', 'Runs_off_Bat'].sum() /
                                     previous_opposition_matches.loc[previous_opposition_matches['Bowling_hand'] == 'Right', 'Balls faced'].sum() * 100) if previous_opposition_matches.loc[previous_opposition_matches['Bowling_hand'] == 'Right', 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_lhbo_opposition = (previous_opposition_matches.loc[previous_opposition_matches['Bowling_hand'] == 'Left', 'Runs_off_Bat'].sum() /
                                     previous_opposition_matches.loc[previous_opposition_matches['Bowling_hand'] == 'Left', 'Balls faced'].sum() * 100) if previous_opposition_matches.loc[previous_opposition_matches['Bowling_hand'] == 'Left', 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_offspin_opposition = (previous_opposition_matches.loc[previous_opposition_matches['Bowling_style'].str.contains('Off spin', na=False), 'Runs_off_Bat'].sum() /
                                        previous_opposition_matches.loc[previous_opposition_matches['Bowling_style'].str.contains('Off spin', na=False), 'Balls faced'].sum() * 100) if previous_opposition_matches.loc[previous_opposition_matches['Bowling_style'].str.contains('Off spin', na=False), 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_legspin_opposition = (previous_opposition_matches.loc[previous_opposition_matches['Bowling_style'].str.contains('Leg spin', na=False), 'Runs_off_Bat'].sum() /
                                        previous_opposition_matches.loc[previous_opposition_matches['Bowling_style'].str.contains('Leg spin', na=False), 'Balls faced'].sum() * 100) if previous_opposition_matches.loc[previous_opposition_matches['Bowling_style'].str.contains('Leg spin', na=False), 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_fastbo_opposition = (previous_opposition_matches.loc[previous_opposition_matches['Bowling_style'].str.contains('Fast', na=False), 'Runs_off_Bat'].sum() /
                                       previous_opposition_matches.loc[previous_opposition_matches['Bowling_style'].str.contains('Fast', na=False), 'Balls faced'].sum() * 100) if previous_opposition_matches.loc[previous_opposition_matches['Bowling_style'].str.contains('Fast', na=False), 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_mediumbo_opposition = (previous_opposition_matches.loc[previous_opposition_matches['Bowling_style'].str.contains('Medium', na=False), 'Runs_off_Bat'].sum() /
                                         previous_opposition_matches.loc[previous_opposition_matches['Bowling_style'].str.contains('Medium', na=False), 'Balls faced'].sum() * 100) if previous_opposition_matches.loc[previous_opposition_matches['Bowling_style'].str.contains('Medium', na=False), 'Balls faced'].sum() > 0 else pd.NA
    cumulative_sr_slowbo_opposition = (previous_opposition_matches.loc[previous_opposition_matches['Bowling_style'].str.contains('Slow', na=False), 'Runs_off_Bat'].sum() /
                                       previous_opposition_matches.loc[previous_opposition_matches['Bowling_style'].str.contains('Slow', na=False), 'Balls faced'].sum() * 100) if previous_opposition_matches.loc[previous_opposition_matches['Bowling_style'].str.contains('Slow', na=False), 'Balls faced'].sum() > 0 else pd.NA
    
    # Store cumulative data in a dictionary and append to list
    cumulative_data_opposition.append({
        'Batsman': batsman,
        'Opposition': opposition,
        'match_id': match_id,
        'No. of Innings_opposition': cumulative_innings_opposition,
        'Runs scored_opposition': cumulative_runs_opposition,
        'Balls faced_opposition': cumulative_balls_opposition,
        'Batting Average_opposition': cumulative_average_opposition,
        'Batting Strike Rate_opposition': cumulative_sr_opposition,
        'Centuries_opposition': (previous_opposition_matches.groupby('match_id')['Runs_off_Bat'].sum() >= 100).sum(),
        'Fifties_opposition': (previous_opposition_matches.groupby('match_id')['Runs_off_Bat'].sum().between(50, 99)).sum(),
        'Sixes_opposition': cumulative_sixes_opposition,
        'Fours_opposition': cumulative_fours_opposition,
        'Highest Score_opposition': cumulative_highest_score_opposition,
        'Batting Strike Rate_Opposition_RHBo': cumulative_sr_rhbo_opposition,
        'Batting Strike Rate_Opposition_LHBo': cumulative_sr_lhbo_opposition,
        'Batting Strike Rate_opposition_OffSpin': cumulative_sr_offspin_opposition,
        'Batting Strike Rate_opposition_LegSpin': cumulative_sr_legspin_opposition,
        'Batting Strike Rate_opposition_FastBo': cumulative_sr_fastbo_opposition,
        'Batting Strike Rate_opposition_MediumBo': cumulative_sr_mediumbo_opposition,
        'Batting Strike Rate_opposition_SlowBo': cumulative_sr_slowbo_opposition
    })

# Convert the list of dictionaries to a DataFrame
cumulative_df_opposition = pd.DataFrame(cumulative_data_opposition)

# Save the updated cumulative stats back to CSV
cumulative_df_opposition.to_csv('cumulative_batsman_stats_opposition.csv', index=False)

# Display the first few rows of the cumulative_df to verify
print(cumulative_df_opposition.head())


       Batsman    Opposition     match_id  No. of Innings_opposition  \
0   A Athanaze  South Africa  1433364.csv                          0   
1  A Dananjaya   Afghanistan  1416079.csv                          0   
2  A Dananjaya    Bangladesh  1133822.csv                          0   
3  A Dananjaya       England  1249207.csv                          0   
4  A Dananjaya         India  1122729.csv                          0   

   Runs scored_opposition  Balls faced_opposition Batting Average_opposition  \
0                       0                       0                       <NA>   
1                       0                       0                       <NA>   
2                       0                       0                       <NA>   
3                       0                       0                       <NA>   
4                       0                       0                       <NA>   

  Batting Strike Rate_opposition  Centuries_opposition  Fifties_opposition  \
0       

In [210]:
import pandas as pd

# Load the cleaned combined dataset
df = pd.read_csv('cleaned_combined_dataset.csv')

# Convert necessary columns to the appropriate data types for calculations
df['Runs_off_Bat'] = pd.to_numeric(df['Runs_off_Bat'], errors='coerce')
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Add 'Balls faced' column assuming each row represents a ball faced
df['Balls faced'] = 1

# Sort data by batsman, date, and match_id for accurate processing
df = df.sort_values(by=['Batsman', 'date', 'match_id']).reset_index(drop=True)

# Initialize a new DataFrame to store cumulative stats up to each match for the last 5 matches
cumulative_df_last5 = []

# Loop over each batsman to calculate cumulative stats up to each match in the last 5 matches
for batsman, batsman_data in df.groupby('Batsman'):
    # Sort batsman's data by date and match_id
    batsman_data = batsman_data.sort_values(by=['date', 'match_id']).reset_index(drop=True)
    
    # Get unique match IDs in reverse chronological order
    match_ids = batsman_data['match_id'].unique()

    # Process each match, considering only the last 5 matches
    for i in range(len(match_ids)):
        # Get the last 5 unique match IDs up to the current match
        last_5_match_ids = match_ids[max(0, i - 5):i]

        # Filter the data for these last 5 matches
        last_5_matches = batsman_data[batsman_data['match_id'].isin(last_5_match_ids)]

        # Calculate cumulative stats for these 5 matches
        cumulative_runs_last5 = last_5_matches['Runs_off_Bat'].sum()
        cumulative_balls_last5 = last_5_matches['Balls faced'].sum()
        cumulative_innings_last5 = last_5_matches['match_id'].nunique()

        cumulative_sixes_last5 = (last_5_matches['Runs_off_Bat'] == 6).sum()
        cumulative_fours_last5 = (last_5_matches['Runs_off_Bat'] == 4).sum()
        cumulative_highest_score_last5 = last_5_matches.groupby('match_id')['Runs_off_Bat'].sum().max() if not last_5_matches.empty else pd.NA

        dismissals_last5 = last_5_matches['Kind_of_wicket_If_Any'].notnull().sum()
        cumulative_average_last5 = cumulative_runs_last5 / dismissals_last5 if dismissals_last5 > 0 else pd.NA

        cumulative_sr_last5 = (cumulative_runs_last5 / cumulative_balls_last5 * 100) if cumulative_balls_last5 > 0 else pd.NA

        # Batting Strike Rate against different types of bowlers in the last 5 matches
        cumulative_sr_rhbo_last5 = (last_5_matches.loc[last_5_matches['Bowling_hand'] == 'Right', 'Runs_off_Bat'].sum() /
                                    last_5_matches.loc[last_5_matches['Bowling_hand'] == 'Right', 'Balls faced'].sum() * 100) if last_5_matches.loc[last_5_matches['Bowling_hand'] == 'Right', 'Balls faced'].sum() > 0 else pd.NA
        cumulative_sr_lhbo_last5 = (last_5_matches.loc[last_5_matches['Bowling_hand'] == 'Left', 'Runs_off_Bat'].sum() /
                                    last_5_matches.loc[last_5_matches['Bowling_hand'] == 'Left', 'Balls faced'].sum() * 100) if last_5_matches.loc[last_5_matches['Bowling_hand'] == 'Left', 'Balls faced'].sum() > 0 else pd.NA
        cumulative_sr_offspin_last5 = (last_5_matches.loc[last_5_matches['Bowling_style'].str.contains('Off spin', na=False), 'Runs_off_Bat'].sum() /
                                       last_5_matches.loc[last_5_matches['Bowling_style'].str.contains('Off spin', na=False), 'Balls faced'].sum() * 100) if last_5_matches.loc[last_5_matches['Bowling_style'].str.contains('Off spin', na=False), 'Balls faced'].sum() > 0 else pd.NA
        cumulative_sr_legspin_last5 = (last_5_matches.loc[last_5_matches['Bowling_style'].str.contains('Leg spin', na=False), 'Runs_off_Bat'].sum() /
                                       last_5_matches.loc[last_5_matches['Bowling_style'].str.contains('Leg spin', na=False), 'Balls faced'].sum() * 100) if last_5_matches.loc[last_5_matches['Bowling_style'].str.contains('Leg spin', na=False), 'Balls faced'].sum() > 0 else pd.NA
        cumulative_sr_fastbo_last5 = (last_5_matches.loc[last_5_matches['Bowling_style'].str.contains('Fast', na=False), 'Runs_off_Bat'].sum() /
                                      last_5_matches.loc[last_5_matches['Bowling_style'].str.contains('Fast', na=False), 'Balls faced'].sum() * 100) if last_5_matches.loc[last_5_matches['Bowling_style'].str.contains('Fast', na=False), 'Balls faced'].sum() > 0 else pd.NA
        cumulative_sr_mediumbo_last5 = (last_5_matches.loc[last_5_matches['Bowling_style'].str.contains('Medium', na=False), 'Runs_off_Bat'].sum() /
                                        last_5_matches.loc[last_5_matches['Bowling_style'].str.contains('Medium', na=False), 'Balls faced'].sum() * 100) if last_5_matches.loc[last_5_matches['Bowling_style'].str.contains('Medium', na=False), 'Balls faced'].sum() > 0 else pd.NA
        cumulative_sr_slowbo_last5 = (last_5_matches.loc[last_5_matches['Bowling_style'].str.contains('Slow', na=False), 'Runs_off_Bat'].sum() /
                                      last_5_matches.loc[last_5_matches['Bowling_style'].str.contains('Slow', na=False), 'Balls faced'].sum() * 100) if last_5_matches.loc[last_5_matches['Bowling_style'].str.contains('Slow', na=False), 'Balls faced'].sum() > 0 else pd.NA

        # Store cumulative data in a dictionary and append to list
        cumulative_df_last5.append({
            'Batsman': batsman,
            'match_id': batsman_data.loc[batsman_data['match_id'] == match_ids[i], 'match_id'].iloc[0],
            'date': batsman_data.loc[batsman_data['match_id'] == match_ids[i], 'date'].iloc[0],
            'No. of Innings_Last5matches': min(cumulative_innings_last5, 5),  # Ensure max of 5 innings
            'Runs scored_Last5matches': cumulative_runs_last5,
            'Balls faced_Last5matches': cumulative_balls_last5,
            'Batting Average_Last5matches': cumulative_average_last5,
            'Batting Strike Rate_Last5matches': cumulative_sr_last5,
            'Centuries_Last5matches': (last_5_matches.groupby('match_id')['Runs_off_Bat'].sum() >= 100).sum(),
            'Fifties_Last5matches': (last_5_matches.groupby('match_id')['Runs_off_Bat'].sum().between(50, 99)).sum(),
            'Sixes_Last5matches': cumulative_sixes_last5,
            'Fours_Last5matches': cumulative_fours_last5,
            'Highest Score_Last5matches': cumulative_highest_score_last5,
            'Batting Strike Rate_Last5matches_RHBo': cumulative_sr_rhbo_last5,
            'Batting Strike Rate_Last5matches_LHBo': cumulative_sr_lhbo_last5,
            'Batting Strike Rate_Last5matches_OffSpin': cumulative_sr_offspin_last5,
            'Batting Strike Rate_Last5matches_LegSpin': cumulative_sr_legspin_last5,
            'Batting Strike Rate_Last5matches_FastBo': cumulative_sr_fastbo_last5,
            'Batting Strike Rate_Last5matches_MediumBo': cumulative_sr_mediumbo_last5,
            'Batting Strike Rate_Last5matches_SlowBo': cumulative_sr_slowbo_last5
        })

# Convert the list of dictionaries to a DataFrame
cumulative_df_last5 = pd.DataFrame(cumulative_df_last5)

# Save the updated cumulative stats back to CSV
cumulative_df_last5.to_csv('cumulative_batsman_stats_last5.csv', index=False)

# Display the first few rows of the cumulative_df to verify
print(cumulative_df_last5.head())


       Batsman     match_id       date  No. of Innings_Last5matches  \
0   A Athanaze  1433364.csv 2024-05-26                            0   
1  A Dananjaya  1122729.csv 2017-12-20                            0   
2  A Dananjaya  1122730.csv 2017-12-22                            1   
3  A Dananjaya  1122731.csv 2017-12-24                            2   
4  A Dananjaya  1133820.csv 2018-03-12                            3   

   Runs scored_Last5matches  Balls faced_Last5matches  \
0                         0                         0   
1                         0                         0   
2                         7                         8   
3                        12                        12   
4                        23                        19   

  Batting Average_Last5matches Batting Strike Rate_Last5matches  \
0                         <NA>                             <NA>   
1                         <NA>                             <NA>   
2                          7.

In [211]:
import pandas as pd

# Load the datasets
career_df = pd.read_csv('cumulative_batsman_stats_career.csv')
ground_df = pd.read_csv('cumulative_batsman_stats_ground.csv')
opposition_df = pd.read_csv('cumulative_batsman_stats_opposition.csv')
cumulative_df = pd.read_csv('cumulative_batsman_stats.csv')
last5_df = pd.read_csv('cumulative_batsman_stats_last5.csv')

# Merging the datasets on 'Batsman' and 'match_id' columns
final_cum_df = cumulative_df.merge(career_df, on=['Batsman', 'match_id'], how='left')\
                         .merge(ground_df, on=['Batsman', 'match_id'], how='left')\
                         .merge(opposition_df, on=['Batsman', 'match_id'], how='left')\
                         .merge(last5_df, on=['Batsman', 'match_id'], how='left')
                        

# Display the first few rows of the merged dataset to verify
print(final_cum_df.head())


       Batsman     match_id  Innings  Runs scored  Balls faced  \
0   A Athanaze  1433364.csv        2            6            8   
1  A Dananjaya  1122729.csv        2            7            8   
2  A Dananjaya  1122730.csv        2            5            4   
3  A Dananjaya  1122731.csv        1           11            7   
4  A Dananjaya  1133820.csv        1            5           11   

   Batting Strike Rate (SR)  No. of Sixes  No. of Fours Batting hand  \
0                 75.000000             0             0         Left   
1                 87.500000             0             0         Left   
2                125.000000             0             1         Left   
3                157.142857             0             2         Left   
4                 45.454545             0             0         Left   

                   Ground  ... Sixes_Last5matches Fours_Last5matches  \
0             Sabina Park  ...                  0                  0   
1        Barabati Stadium 

In [212]:
career_count = career_df.shape[0]
ground_count = ground_df.shape[0]
opposition_count = opposition_df.shape[0]
cumulative_count = cumulative_df.shape[0]
last5_count =  last5_df.shape[0]
merged_count = final_cum_df.shape[0]
# Print the row counts
print(f"Career dataset row count: {career_count}")
print(f"Ground dataset row count: {ground_count}")
print(f"Opposition dataset row count: {opposition_count}")
print(f"Cumulative dataset row count: {cumulative_count}")
print(f"Cumulative dataset row count: {last5_count}")
print(f"Merged dataset row count: {merged_count}")

Career dataset row count: 13050
Ground dataset row count: 13050
Opposition dataset row count: 13050
Cumulative dataset row count: 13050
Cumulative dataset row count: 13050
Merged dataset row count: 13050


In [213]:
# Get the count of missing values in each column
missing_values_count = final_cum_df.isnull().sum()
missing_values_count
#merged_df.to_csv('final_cumulative_batsman_stats.csv', index=False)

Batsman                                          0
match_id                                         0
Innings                                          0
Runs scored                                      0
Balls faced                                      0
                                             ...  
Batting Strike Rate_Last5matches_OffSpin      2298
Batting Strike Rate_Last5matches_LegSpin      4357
Batting Strike Rate_Last5matches_FastBo       1125
Batting Strike Rate_Last5matches_MediumBo     1856
Batting Strike Rate_Last5matches_SlowBo      13042
Length: 92, dtype: int64

In [214]:
final_cum_zeros_df =  final_cum_df.fillna(0, inplace=True)

In [215]:
missing_values_count = final_cum_df.isnull().sum()
missing_values_count

Batsman                                      0
match_id                                     0
Innings                                      0
Runs scored                                  0
Balls faced                                  0
                                            ..
Batting Strike Rate_Last5matches_OffSpin     0
Batting Strike Rate_Last5matches_LegSpin     0
Batting Strike Rate_Last5matches_FastBo      0
Batting Strike Rate_Last5matches_MediumBo    0
Batting Strike Rate_Last5matches_SlowBo      0
Length: 92, dtype: int64

In [217]:
# Sort the dataframe by Batsman and match_id
final_cum_df.sort_values(by=['Batsman', 'match_id'], inplace=True)

# Shift the 'Runs scored' column to get the next match runs
final_cum_df['next_match_runs'] = final_cum_df.groupby('Batsman')['Runs scored'].shift(-1)

# Fill any NaN values in 'next_match_runs' with 0
final_cum_df['next_match_runs'] = final_cum_df['next_match_runs'].fillna(0)

# Drop the specified columns
final_cum_df.drop(columns=['Ground', 'Bowling_Team'], inplace=True)

# Verify the result
print(final_cum_df.head())

       Batsman     match_id  Innings  Runs scored  Balls faced  \
0   A Athanaze  1433364.csv        2            6            8   
1  A Dananjaya  1122729.csv        2            7            8   
2  A Dananjaya  1122730.csv        2            5            4   
3  A Dananjaya  1122731.csv        1           11            7   
4  A Dananjaya  1133820.csv        1            5           11   

   Batting Strike Rate (SR)  No. of Sixes  No. of Fours Batting hand   Toss  \
0                 75.000000             0             0         Left    bat   
1                 87.500000             0             0         Left  field   
2                125.000000             0             1         Left  field   
3                157.142857             0             2         Left  field   
4                 45.454545             0             0         Left  field   

   ... Fours_Last5matches Highest Score_Last5matches  \
0  ...                  0                        0.0   
1  ...          

In [218]:
final_cum_df.drop(columns=['date', 'venue'], inplace=True)

In [219]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

# Assuming merged_df is your dataset
merged_df = final_cum_df.drop(columns=['Batsman', 'match_id'])
merged_df = pd.get_dummies(merged_df)

X = merged_df.drop(columns=['next_match_runs'])
y = merged_df['next_match_runs']

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply PCA
pca = PCA(n_components=0.95)  # Keep 95% of the variance
X_pca = pca.fit_transform(X_scaled)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=42)

# Initialize the linear regression model
model = LinearRegression()

# Train the model
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

# Print the results
print(f"Mean Squared Error (MSE): {mse}")
print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"R^2 Score: {r2}")
print(f"Mean Absolute Error (MAE): {mae}")

# Optional: Print PCA variance ratios
print(f"Explained Variance Ratio by Each Principal Component: {pca.explained_variance_ratio_}")
print(f"Total Explained Variance Ratio: {sum(pca.explained_variance_ratio_)}")


Mean Squared Error (MSE): 380.4850574018755
Root Mean Squared Error (RMSE): 19.50602618171819
R^2 Score: 0.04794328822169902
Mean Absolute Error (MAE): 14.571214149621586
Explained Variance Ratio by Each Principal Component: [0.03014204 0.00941531 0.00750813 0.00685842 0.00423086 0.00357667
 0.00292014 0.00265346 0.00248411 0.00245456 0.00238057 0.00228811
 0.00227126 0.00218582 0.00216453 0.00213186 0.00209275 0.00208166
 0.00206095 0.00204051 0.00194002 0.00183781 0.00177419 0.00169436
 0.00166462 0.00161457 0.00159341 0.00156885 0.00152449 0.00149834
 0.00149442 0.00146811 0.00145116 0.00142136 0.00140668 0.00140341
 0.00138221 0.00135519 0.00133722 0.00132617 0.0013048  0.00129995
 0.00129886 0.00129533 0.00128421 0.00127772 0.0012732  0.0012712
 0.0012678  0.00126548 0.00126451 0.00125789 0.00124959 0.00124679
 0.00124471 0.0012419  0.00124136 0.00123957 0.00123797 0.00123758
 0.00123546 0.00123441 0.00123375 0.0012323  0.00123136 0.00123061
 0.00122965 0.00122865 0.00122793 0.001